# 모델 성능 분석 및 시각화
## 공간별 (livingroom, room1, room2) 모델 성능 비교

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (15, 8)
sns.set_style('whitegrid')

In [ ]:
# 데이터 경로 설정
base_path = Path('version2_250813_processed')
rooms = ['livingroom', 'room1', 'room2']
models = ['DLinear', 'FreTS', 'iTransformer', 'Koopa', 'MICN', 
          'PatchTST', 'TimeMixer', 'TimesNet', 'TimeXer', 'TSMixer']
systems = {
    'KETI_system_control': '환기유효전력',
    'system_control': '냉난방유효전력'
}

In [ ]:
def load_test_results(room, model, system):
    """test_results.json 파일 로드"""
    file_path = base_path / room / model / system / 'version0' / 'test_results.json'
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        return data
    except FileNotFoundError:
        print(f"파일을 찾을 수 없습니다: {file_path}")
        return None
    except json.JSONDecodeError:
        print(f"JSON 파싱 오류: {file_path}")
        return None

In [ ]:
# 모든 데이터 수집
all_results = []

for room in rooms:
    for model in models:
        for system_key, system_name in systems.items():
            result = load_test_results(room, model, system_key)
            if result:
                result['room'] = room
                result['model'] = model
                result['system'] = system_name
                result['system_key'] = system_key
                all_results.append(result)

# DataFrame 생성
df = pd.DataFrame(all_results)
print(f"총 {len(df)}개의 결과 로드됨")
print("\n데이터 샘플:")
df.head()

In [ ]:
# Infinity 값 처리
df['rse'] = df['rse'].replace([np.inf, -np.inf], np.nan)
df['nmae'] = df['nmae'].replace([np.inf, -np.inf], np.nan)

# 기본 통계
print("\n=== 기본 통계 ===")
print(df.groupby(['room', 'system'])[['mse', 'mae', 'rmse', 'mape']].mean())

## 1. 공간별 모델 성능 비교 (MSE, MAE, RMSE)

In [ ]:
metrics = ['mse', 'mae', 'rmse', 'mape']
metric_names = {'mse': 'MSE', 'mae': 'MAE', 'rmse': 'RMSE', 'mape': 'MAPE (%)'}

for metric in metrics:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'{metric_names[metric]} - 공간별 모델 성능 비교', fontsize=16, fontweight='bold')
    
    for idx, room in enumerate(rooms):
        ax = axes[idx]
        room_data = df[df['room'] == room]
        
        # 시스템별로 그룹화하여 플롯
        x = np.arange(len(models))
        width = 0.35
        
        ventilation_data = room_data[room_data['system'] == '환기유효전력'].sort_values('model')
        hvac_data = room_data[room_data['system'] == '냉난방유효전력'].sort_values('model')
        
        ax.bar(x - width/2, ventilation_data[metric].values, width, label='환기유효전력', alpha=0.8)
        ax.bar(x + width/2, hvac_data[metric].values, width, label='냉난방유효전력', alpha=0.8)
        
        ax.set_xlabel('Model', fontsize=12)
        ax.set_ylabel(metric_names[metric], fontsize=12)
        ax.set_title(f'{room.upper()}', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(sorted(models), rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 2. Prediction Accuracy 비교

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Prediction Accuracy - 공간별 모델 성능 비교', fontsize=16, fontweight='bold')

for idx, room in enumerate(rooms):
    ax = axes[idx]
    room_data = df[df['room'] == room]
    
    x = np.arange(len(models))
    width = 0.35
    
    ventilation_data = room_data[room_data['system'] == '환기유효전력'].sort_values('model')
    hvac_data = room_data[room_data['system'] == '냉난방유효전력'].sort_values('model')
    
    ax.bar(x - width/2, ventilation_data['prediction_accuracy'].values * 100, width, 
           label='환기유효전력', alpha=0.8)
    ax.bar(x + width/2, hvac_data['prediction_accuracy'].values * 100, width, 
           label='냉난방유효전력', alpha=0.8)
    
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title(f'{room.upper()}', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(sorted(models), rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([90, 100])  # Accuracy는 보통 높으므로 범위 조정

plt.tight_layout()
plt.show()

## 3. 히트맵: 모델별 성능 비교

In [ ]:
for system_name in ['환기유효전력', '냉난방유효전력']:
    system_data = df[df['system'] == system_name]
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.suptitle(f'{system_name} - 모델 성능 히트맵 (공간별)', fontsize=16, fontweight='bold')
    
    for idx, metric in enumerate(['mse', 'mae', 'rmse', 'mape']):
        ax = axes[idx // 2, idx % 2]
        
        # 피벗 테이블 생성
        pivot_data = system_data.pivot_table(
            values=metric, 
            index='model', 
            columns='room'
        )
        
        # 히트맵 생성
        sns.heatmap(pivot_data, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax, 
                    cbar_kws={'label': metric_names[metric]})
        ax.set_title(f'{metric_names[metric]}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Room', fontsize=12)
        ax.set_ylabel('Model', fontsize=12)
    
    plt.tight_layout()
    plt.show()

## 4. 최고 성능 모델 분석

In [ ]:
print("\n=== 최고 성능 모델 (MSE 기준) ===")
for room in rooms:
    print(f"\n{room.upper()}:")
    for system_name in ['환기유효전력', '냉난방유효전력']:
        filtered_df = df[(df['room'] == room) & (df['system'] == system_name)]
        best_model = filtered_df.loc[filtered_df['mse'].idxmin()]
        print(f"  {system_name}: {best_model['model']} (MSE: {best_model['mse']:.6f}, MAE: {best_model['mae']:.4f})")

In [ ]:
print("\n=== 최고 성능 모델 (Accuracy 기준) ===")
for room in rooms:
    print(f"\n{room.upper()}:")
    for system_name in ['환기유효전력', '냉난방유효전력']:
        filtered_df = df[(df['room'] == room) & (df['system'] == system_name)]
        best_model = filtered_df.loc[filtered_df['prediction_accuracy'].idxmax()]
        print(f"  {system_name}: {best_model['model']} (Accuracy: {best_model['prediction_accuracy']*100:.2f}%)")

## 5. 시스템별 전체 성능 비교

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('환기유효전력 vs 냉난방유효전력 - 전체 모델 성능 비교', fontsize=16, fontweight='bold')

for idx, metric in enumerate(['mse', 'mae', 'rmse', 'mape']):
    ax = axes[idx // 2, idx % 2]
    
    # 시스템별 평균 계산
    system_avg = df.groupby(['system', 'model'])[metric].mean().reset_index()
    
    # 피벗
    pivot_data = system_avg.pivot(index='model', columns='system', values=metric)
    
    x = np.arange(len(pivot_data))
    width = 0.35
    
    ax.bar(x - width/2, pivot_data['환기유효전력'].values, width, 
           label='환기유효전력', alpha=0.8)
    ax.bar(x + width/2, pivot_data['냉난방유효전력'].values, width, 
           label='냉난방유효전력', alpha=0.8)
    
    ax.set_xlabel('Model', fontsize=12)
    ax.set_ylabel(metric_names[metric], fontsize=12)
    ax.set_title(f'{metric_names[metric]} (전체 평균)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot_data.index, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 상세 성능 테이블

In [ ]:
# 각 공간별로 상세 테이블 생성
for room in rooms:
    print(f"\n{'='*80}")
    print(f"{room.upper()} - 상세 성능 테이블")
    print(f"{'='*80}")
    
    room_data = df[df['room'] == room][['model', 'system', 'mse', 'mae', 'rmse', 'mape', 'prediction_accuracy']].copy()
    room_data['prediction_accuracy'] = room_data['prediction_accuracy'] * 100
    room_data = room_data.sort_values(['system', 'mse'])
    
    print(room_data.to_string(index=False))

## 7. 모델 랭킹

In [ ]:
# MSE 기준 랭킹
print("\n=== MSE 기준 모델 랭킹 (낮을수록 좋음) ===")
for room in rooms:
    print(f"\n{room.upper()}:")
    for system_name in ['환기유효전력', '냉난방유효전력']:
        print(f"\n  {system_name}:")
        filtered_df = df[(df['room'] == room) & (df['system'] == system_name)].copy()
        filtered_df = filtered_df.sort_values('mse')
        for idx, row in enumerate(filtered_df.itertuples(), 1):
            print(f"    {idx}. {row.model}: MSE={row.mse:.6f}, MAE={row.mae:.4f}, Accuracy={row.prediction_accuracy*100:.2f}%")

In [ ]:
# 데이터 저장
output_file = 'model_performance_summary.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n결과가 '{output_file}'에 저장되었습니다.")